- loops
- rename
- apply
- aggregate

In [18]:
import pandas as pd

# data = "https://raw.githubusercontent.com/nkmwicz/data-for-students/refs/heads/main/french-ambs.csv"
# df = pd.read_csv(data)
# df.head()

In [19]:
# ./filename.csv call path at current folder.filename.csv
# ../ => goes back one folder
# df.to_csv("../data/french-ambs.csv", index=False)

In [20]:
df = pd.read_csv("../data/french-ambs.csv")
df.head()

,id,name,place,year,lat,long,lat_long
0,0,"Richer, Christophe",Denmark,1543,55.68,12.57,"55.68, 12.57"
1,1,"Richer, Christophe",Denmark,1541,55.68,12.57,"55.68, 12.57"
2,2,"Danzay, Charles de",Denmark,1561,55.68,12.57,"55.68, 12.57"
3,3,"Danzay, Charles de",Denmark,1563,55.68,12.57,"55.68, 12.57"
4,4,"Danzay, Charles de",Denmark,1582,55.68,12.57,"55.68, 12.57"


In [21]:
df = df.drop(columns=["lat_long"])
df.head()

,id,name,place,year,lat,long
0,0,"Richer, Christophe",Denmark,1543,55.68,12.57
1,1,"Richer, Christophe",Denmark,1541,55.68,12.57
2,2,"Danzay, Charles de",Denmark,1561,55.68,12.57
3,3,"Danzay, Charles de",Denmark,1563,55.68,12.57
4,4,"Danzay, Charles de",Denmark,1582,55.68,12.57


In [22]:
df = df[["name", "place", "year"]]
df.head()

,name,place,year
0,"Richer, Christophe",Denmark,1543
1,"Richer, Christophe",Denmark,1541
2,"Danzay, Charles de",Denmark,1561
3,"Danzay, Charles de",Denmark,1563
4,"Danzay, Charles de",Denmark,1582


In [23]:
df["place"].unique()

<StringArray>
[          'Denmark',           'England',            'Geneva',
           'Grisons', 'Holy Roman Empire',       'Netherlands',
    'Ottoman Empire',          'Portugal',              'Rome',
          'Scotland',             'Spain',     'Swiss Cantons',
            'Venice',            'Poland',             'Savoy',
           'Ferrara',            'Saxony',           'Tuscany']
Length: 18, dtype: str

In [24]:
df["place"].count()
df["place"].nunique()

18

In [25]:
# Loops
# iterrows / itertuples
# iterrows
for idx, row in df.iterrows():
    if row["place"] == "Holy Roman Empire":
        df.at[idx, "place"] = "HRE"

df["place"].unique()

<StringArray>
[       'Denmark',        'England',         'Geneva',        'Grisons',
            'HRE',    'Netherlands', 'Ottoman Empire',       'Portugal',
           'Rome',       'Scotland',          'Spain',  'Swiss Cantons',
         'Venice',         'Poland',          'Savoy',        'Ferrara',
         'Saxony',        'Tuscany']
Length: 18, dtype: str

In [26]:
for row in df.itertuples():
    idx = row.Index
    if row.place == "Ottoman Empire":
        df.at[idx,"place"] = "Ott"

df["place"].unique()

<StringArray>
[      'Denmark',       'England',        'Geneva',       'Grisons',
           'HRE',   'Netherlands',           'Ott',      'Portugal',
          'Rome',      'Scotland',         'Spain', 'Swiss Cantons',
        'Venice',        'Poland',         'Savoy',       'Ferrara',
        'Saxony',       'Tuscany']
Length: 18, dtype: str

In [27]:
# using .loc to replace

df.loc[df["place"] == "Netherlands", "place"] = "Neth"
df["place"].unique()

<StringArray>
[      'Denmark',       'England',        'Geneva',       'Grisons',
           'HRE',          'Neth',           'Ott',      'Portugal',
          'Rome',      'Scotland',         'Spain', 'Swiss Cantons',
        'Venice',        'Poland',         'Savoy',       'Ferrara',
        'Saxony',       'Tuscany']
Length: 18, dtype: str

In [28]:
# replace many with a  hash using .replace
hash = {place: f"{place[:3] + '.' if len(place) > 4 else place}" for place in df["place"].unique()}
df["place"] = df["place"].replace(hash)

In [29]:
df["place"].unique()

<StringArray>
['Den.', 'Eng.', 'Gen.', 'Gri.',  'HRE', 'Neth',  'Ott', 'Por.', 'Rome',
 'Sco.', 'Spa.', 'Swi.', 'Ven.', 'Pol.', 'Sav.', 'Fer.', 'Sax.', 'Tus.']
Length: 18, dtype: str

In [30]:
# complicated boolean logic with | (or), & (and), ~ (not)
df.loc[((df["place"] == "Netherlands") | (df["place"] == "England")) & (df["year"] < 1550)]

,name,place,year


In [31]:
df.loc[~((df["place"] == "Netherlands") | (df["place"] == "England")) & (df["year"] < 1550)]

,name,place,year
0,"Richer, Christophe",Den.,1543
1,"Richer, Christophe",Den.,1541
5,"Bapaume, Robert de",Eng.,1516
6,"Passano, Jean-Joachim de",Eng.,1527
7,De Monpreat,Eng.,1532
...,...,...,...
854,"Daugerant, Louis",Swi.,1537
861,"Montluc, Jean de",Ven.,1542
864,"Canossa, Louis de",Ven.,1526
866,"De Selve, Georges",Ven.,1537


In [32]:
# .apply() takes a function
df["year_long"] = df["year"].apply(lambda val: str(val) + " CE")
df["year_long"]

0      1543 CE
1      1541 CE
2      1561 CE
3      1563 CE
4      1582 CE
        ...   
864    1526 CE
865    1597 CE
866    1537 CE
867    1559 CE
868    1525 CE
Name: year_long, Length: 869, dtype: str

In [37]:
def get_total_years(row):
    name = row['name']
    pl = row['place']
    num_years = df.loc[(df['name'] == name) & (df['place'] == pl), "name"].count()
    return num_years
    
df["years_present"] = df.apply(get_total_years, axis=1)
df.head()

,name,place,year,year_long,years_present
0,"Richer, Christophe",Den.,1543,1543 CE,7
1,"Richer, Christophe",Den.,1541,1541 CE,7
2,"Danzay, Charles de",Den.,1561,1561 CE,33
3,"Danzay, Charles de",Den.,1563,1563 CE,33
4,"Danzay, Charles de",Den.,1582,1582 CE,33


In [40]:
df_g = df.groupby(["name"]).agg({
    "place": "nunique",
    "years_present": "count",
    "year": "min",
    "year": "max",
})
df_g

,place,years_present,year
name,,,
"Adhemar, Louis",1,2,1539
"Ancel, Guillaume",1,25,1600
"Babou, Philibert",1,4,1561
"Bapaume, Robert de",1,1,1516
"Bellièvre, Jean de, sr. de Hautefort",1,7,1579
...,...,...,...
"d'Armagnac, Georges",2,7,1544
"d'Ebrard de Saint Sulpice, Jean",1,4,1565
"d'Esneval, Baron",1,2,1586


In [43]:
df_g = df.groupby(["name"]).agg(
    num_places=("place", "nunique"),
    locations=("place", "unique"),
    years_abroad=("years_present", "count"), 
    year_start=("year", "min"),
    year_end=("year", "max"),
).reset_index()
df_g

,name,num_places,locations,years_abroad,year_start,year_end
0,"Adhemar, Louis",1,[Rome],2,1538,1539
1,"Ancel, Guillaume",1,[HRE],25,1576,1600
2,"Babou, Philibert",1,[Rome],4,1558,1561
3,"Bapaume, Robert de",1,[Eng.],1,1516,1516
4,"Bellièvre, Jean de, sr. de Hautefort",1,[Swi.],7,1573,1579
...,...,...,...,...,...,...
149,"d'Armagnac, Georges",2,"[Rome, Ven.]",7,1536,1544
150,"d'Ebrard de Saint Sulpice, Jean",1,[Spa.],4,1562,1565
151,"d'Esneval, Baron",1,[Sco.],2,1585,1586
152,"d'Urfé, Claude",1,[Rome],4,1548,1551


In [47]:
df_g1 = df.groupby(["name", "place"]).agg(
    years_abroad=("years_present", "count"), 
    year_start=("year", "min"),
    year_end=("year", "max"),
).reset_index().sort_values(by="name")
df_g1

,name,place,years_abroad,year_start,year_end
0,"Adhemar, Louis",Rome,2,1538,1539
1,"Ancel, Guillaume",HRE,25,1576,1600
2,"Babou, Philibert",Rome,4,1558,1561
3,"Bapaume, Robert de",Eng.,1,1516,1516
4,"Bellièvre, Jean de, sr. de Hautefort",Swi.,7,1573,1579
...,...,...,...,...,...
181,"d'Armagnac, Georges",Ven.,3,1536,1538
182,"d'Ebrard de Saint Sulpice, Jean",Spa.,4,1562,1565
183,"d'Esneval, Baron",Sco.,2,1585,1586
184,"d'Urfé, Claude",Rome,4,1548,1551
